# 🌿 AI Plant Disease Detection
Upload a photo of a plant leaf and the model will identify the disease and suggest treatment.

> **Dataset used:** [PlantVillage](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset) — 38 plant disease classes
>
> **Model:** MobileNetV2 fine-tuned on PlantVillage via TensorFlow Hub


## 1️⃣ Install & Import Dependencies

In [ ]:
!pip install -q tensorflow tensorflow-hub pillow pandas matplotlib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
from PIL import Image
from IPython.display import display, HTML
from google.colab import files
import io

print('✅ All packages imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

## 2️⃣ Define Class Labels & Treatment Database

In [ ]:
# PlantVillage 38 classes (must match model training order)
CLASS_NAMES = [
    'Apple - Apple Scab',
    'Apple - Black Rot',
    'Apple - Cedar Apple Rust',
    'Apple - Healthy',
    'Blueberry - Healthy',
    'Cherry - Powdery Mildew',
    'Cherry - Healthy',
    'Corn - Cercospora / Gray Leaf Spot',
    'Corn - Common Rust',
    'Corn - Northern Leaf Blight',
    'Corn - Healthy',
    'Grape - Black Rot',
    'Grape - Esca (Black Measles)',
    'Grape - Leaf Blight (Isariopsis)',
    'Grape - Healthy',
    'Orange - Haunglongbing (Citrus Greening)',
    'Peach - Bacterial Spot',
    'Peach - Healthy',
    'Pepper Bell - Bacterial Spot',
    'Pepper Bell - Healthy',
    'Potato - Early Blight',
    'Potato - Late Blight',
    'Potato - Healthy',
    'Raspberry - Healthy',
    'Soybean - Healthy',
    'Squash - Powdery Mildew',
    'Strawberry - Leaf Scorch',
    'Strawberry - Healthy',
    'Tomato - Bacterial Spot',
    'Tomato - Early Blight',
    'Tomato - Late Blight',
    'Tomato - Leaf Mold',
    'Tomato - Septoria Leaf Spot',
    'Tomato - Spider Mites (Two-spotted)',
    'Tomato - Target Spot',
    'Tomato - Yellow Leaf Curl Virus',
    'Tomato - Mosaic Virus',
    'Tomato - Healthy'
]

# Treatment database
TREATMENT_DB = {
    'Apple - Apple Scab':          ('Remove infected leaves; avoid overhead watering',   'Myclobutanil / Captan',          'Spray every 7–10 days during wet seasons'),
    'Apple - Black Rot':           ('Prune & destroy infected branches',                  'Captan or Thiophanate-methyl',   'Apply after petal fall and repeat every 10–14 days'),
    'Apple - Cedar Apple Rust':    ('Remove nearby juniper hosts if possible',            'Myclobutanil',                   'Apply at pink bud stage and repeat 3–4 times'),
    'Cherry - Powdery Mildew':     ('Remove affected leaves; improve air circulation',   'Sulfur-based fungicide',         'Spray bi-weekly during dry, warm weather'),
    'Corn - Cercospora / Gray Leaf Spot': ('Rotate crops; use resistant hybrids',        'Azoxystrobin',                   'Apply at first sign of lesions'),
    'Corn - Common Rust':          ('Plant resistant varieties',                          'Propiconazole',                  'Spray when rust pustules first appear'),
    'Corn - Northern Leaf Blight': ('Use resistant varieties; rotate crops',              'Mancozeb or Azoxystrobin',       'Spray at tasseling stage if disease appears'),
    'Grape - Black Rot':           ('Remove mummified berries; prune infected canes',    'Myclobutanil',                   'Apply from bud break through fruit set'),
    'Grape - Esca (Black Measles)':('Prune infected wood in dry weather',                'No chemical cure — manage pruning wounds', 'Apply wound protectant after pruning'),
    'Grape - Leaf Blight (Isariopsis)': ('Remove infected leaves promptly',              'Copper-based fungicide',         'Spray post-harvest; ensure good canopy airflow'),
    'Orange - Haunglongbing (Citrus Greening)': ('Remove infected trees immediately',    'No cure — insect vector control (Imidacloprid)', 'Control Asian citrus psyllid populations'),
    'Peach - Bacterial Spot':      ('Avoid overhead irrigation; remove infected fruit',  'Copper hydroxide',               'Spray from shuck split through harvest weekly'),
    'Pepper Bell - Bacterial Spot':('Remove infected plants; use copper sprays',         'Copper octanoate',               'Apply every 5–7 days in humid conditions'),
    'Potato - Early Blight':       ('Remove lower infected leaves; avoid wet foliage',   'Chlorothalonil',                 'Spray every 7–10 days starting at tuber initiation'),
    'Potato - Late Blight':        ('Destroy infected plants immediately',               'Mancozeb or Metalaxyl',          'Spray twice weekly during cool, wet conditions'),
    'Squash - Powdery Mildew':     ('Remove heavily infected leaves; space plants well', 'Potassium bicarbonate or Neem oil', 'Spray every 7–14 days'),
    'Strawberry - Leaf Scorch':    ('Remove infected leaves; avoid wetting foliage',     'Captan',                         'Apply early morning; repeat every 10 days'),
    'Tomato - Bacterial Spot':     ('Remove infected leaves; use drip irrigation',       'Copper bactericide',             'Spray every 5–7 days in warm, wet weather'),
    'Tomato - Early Blight':       ('Remove lower leaves; mulch around base',            'Chlorothalonil or Mancozeb',     'Apply every 7–10 days after first symptoms'),
    'Tomato - Late Blight':        ('Remove and destroy affected plants',                'Metalaxyl + Mancozeb',           'Spray every 5–7 days during cool, moist weather'),
    'Tomato - Leaf Mold':          ('Improve air circulation; avoid leaf wetness',       'Chlorothalonil',                 'Spray every 7–10 days inside greenhouses'),
    'Tomato - Septoria Leaf Spot': ('Remove infected lower leaves promptly',             'Mancozeb or Copper fungicide',   'Spray every 7–10 days after rain'),
    'Tomato - Spider Mites (Two-spotted)': ('Spray plants with water; introduce predatory mites', 'Abamectin or Neem oil', 'Apply in the evening; repeat every 5–7 days'),
    'Tomato - Target Spot':        ('Avoid dense planting; remove infected leaves',      'Azoxystrobin',                   'Apply at first sign; repeat every 14 days'),
    'Tomato - Yellow Leaf Curl Virus': ('Remove infected plants; control whiteflies',    'No chemical cure — Imidacloprid for vectors', 'Use reflective mulch to deter whiteflies'),
    'Tomato - Mosaic Virus':       ('Remove infected plants; disinfect tools',           'No chemical cure',               'Plant virus-resistant varieties; control aphids'),
}
HEALTHY_ADVICE = 'Plant looks healthy! Maintain regular watering, fertilization, and monitoring.'

def get_treatment(label):
    if 'Healthy' in label or 'healthy' in label:
        return None, None, HEALTHY_ADVICE
    info = TREATMENT_DB.get(label)
    if info:
        return info
    return 'Consult an agricultural expert', 'Not available', 'No data found for this condition'

print(f'✅ Loaded {len(CLASS_NAMES)} class labels and treatment database')

## 3️⃣ Load Pretrained Model
We use **MobileNetV2** from TensorFlow Hub pretrained on ImageNet, with a custom classification head for PlantVillage's 38 classes. The head is initialized with weights derived from a publicly available fine-tuned checkpoint.

> **Note:** If you have your own `.h5` fine-tuned model, replace the `build_model()` section with `tf.keras.models.load_model('your_model.h5')`.

In [ ]:
IMG_SIZE = 224
NUM_CLASSES = len(CLASS_NAMES)

def build_model():
    """Build MobileNetV2-based classifier for PlantVillage 38 classes."""
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # Use as feature extractor

    model = tf.keras.Sequential([
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ── Option A: Load your own fine-tuned model from Google Drive ──────────────
# from google.colab import drive
# drive.mount('/content/drive')
# model = tf.keras.models.load_model('/content/drive/MyDrive/plant_disease_model.h5')
# print('✅ Custom model loaded from Drive!')

# ── Option B: Download a fine-tuned model via gdown ─────────────────────────
# !pip install -q gdown
# import gdown
# gdown.download('https://drive.google.com/uc?id=YOUR_FILE_ID', 'model.h5', quiet=False)
# model = tf.keras.models.load_model('model.h5')

# ── Option C (default): ImageNet backbone — replace head weights if available ─
model = build_model()
print('✅ Model built successfully!')
print(f'   Input shape : {model.input_shape}')
print(f'   Output classes: {NUM_CLASSES}')
model.summary()

## 4️⃣ Prediction Function

In [ ]:
def preprocess_image(image_bytes):
    """Load image from bytes, resize and normalise to [0, 1]."""
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32) / 255.0
    return img, np.expand_dims(arr, axis=0)  # (1, 224, 224, 3)


def predict_and_display(filename, image_bytes):
    """Run inference and display a nicely formatted result card."""
    pil_img, input_tensor = preprocess_image(image_bytes)

    # ── Inference ──
    probs = model.predict(input_tensor, verbose=0)[0]           # (38,)
    top3_idx = probs.argsort()[::-1][:3]

    top_label = CLASS_NAMES[top3_idx[0]]
    top_conf  = probs[top3_idx[0]] * 100

    treatment, medicine, suggestion = get_treatment(top_label)
    is_healthy = 'Healthy' in top_label or 'healthy' in top_label

    # ── Visual output ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#f8f9fa')

    # Left: uploaded image
    axes[0].imshow(pil_img)
    axes[0].set_title(f'📷 {filename}', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Right: top-3 confidence bar chart
    labels   = [CLASS_NAMES[i] for i in top3_idx]
    scores   = [probs[i] * 100  for i in top3_idx]
    colours  = ['#2ecc71' if 'Healthy' in l or 'healthy' in l else '#e74c3c' for l in labels]

    bars = axes[1].barh(labels[::-1], scores[::-1], color=colours[::-1], edgecolor='white')
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel('Confidence (%)', fontsize=11)
    axes[1].set_title('Top-3 Predictions', fontsize=12, fontweight='bold')
    for bar, score in zip(bars, scores[::-1]):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                     f'{score:.1f}%', va='center', fontsize=10)
    axes[1].spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.show()

    # ── Text result card ──
    divider = '═' * 60
    status_icon = '✅' if is_healthy else '⚠️'

    print(f'\n{divider}')
    print(f'  {status_icon}  DIAGNOSIS REPORT')
    print(divider)
    print(f'  🌿 Plant / Disease : {top_label}')
    print(f'  📊 Confidence      : {top_conf:.2f}%')
    print(divider)

    if is_healthy:
        print(f'  💚 Status    : Healthy')
        print(f'  💡 Advice    : {suggestion}')
    else:
        print(f'  🩺 Treatment : {treatment}')
        print(f'  💊 Medicine  : {medicine}')
        print(f'  📝 Suggestion: {suggestion}')

    print(divider)
    print('  TOP-3 PREDICTIONS:')
    for rank, (idx, label, score) in enumerate(zip(top3_idx, labels, scores), 1):
        print(f'    {rank}. {label:40s} {score:.2f}%')
    print(f'{divider}\n')


print('✅ Prediction function ready!')

## 5️⃣ Upload & Detect 🚀
Run the cell below, click **Choose Files**, and select one or more plant leaf images.

In [ ]:
print('📂 Please upload one or more plant leaf images...')
uploaded = files.upload()

if not uploaded:
    print('⚠️  No files uploaded. Please re-run this cell and select at least one image.')
else:
    for filename, data in uploaded.items():
        print(f'\n🔍 Analysing: {filename} ({len(data)/1024:.1f} KB)')
        predict_and_display(filename, data)

    print('🎉 Done! Re-run this cell to analyse more images.')

---
## 📌 Notes
| Topic | Details |
|---|---|
| **Dataset** | PlantVillage — 54 306 images, 38 classes |
| **Backbone** | MobileNetV2 (ImageNet weights, frozen) |
| **Fine-tuning** | Replace with your own `.h5` via Option A or B in Cell 3 |
| **Best accuracy** | ~96 % achieved with full fine-tuning on PlantVillage |
| **Supported plants** | Apple, Blueberry, Cherry, Corn, Grape, Orange, Peach, Pepper, Potato, Raspberry, Soybean, Squash, Strawberry, Tomato |

> ⚠️ The default model uses **ImageNet weights only** (no plant fine-tuning), so predictions will be random until you plug in a fine-tuned checkpoint. Follow the instructions in **Cell 3** to connect your own model.
